In [2]:

import os
import sys

pid = os.fork()

if pid == 0:
    # Child process
    print(f"Child Process (PID: {os.getpid()}) running 'ls -l'...")
    os.execvp("ls", ["ls", "-l"])
    sys.exit(1)

elif pid > 0:
    # Parent process
    print(f"Parent Process (PID: {os.getpid()}) waiting for child (PID: {pid})...")
    child_pid, status = os.wait()
    print(f"Parent: Child (PID: {child_pid}) finished with exit code {os.waitstatus_to_exitcode(status)}.")

Child Process (PID: 5478) running 'ls -l'...
Parent Process (PID: 4728) waiting for child (PID: 5478)...
Parent: Child (PID: 5478) finished with exit code 0.


/tmp/ipykernel_4728/1215317868.py:4: DeprecationWarning: This process (pid=4728) is multi-threaded, use of fork() may lead to deadlocks in the child.
  pid = os.fork()


In [3]:
import threading
import time

def task(num):
    print("Thread", num, "started")

    time.sleep(2)

    print("Thread", num, "finished")

threads = []

for i in range(1, 4):
    t = threading.Thread(target=task, args=(i,))
    threads.append(t)
    t.start()

for t in threads:
    t.join()

print("All threads completed.")

Thread 1 started
Thread 2 started
Thread 3 started
Thread 1 finished
Thread 3 finished
Thread 2 finished
All threads completed.


In [4]:
def calculate(processes, method, quantum=2):
    n = len(processes)
    remaining = {p[0]: p[2] for p in processes}
    completion = {}
    time = 0
    order = []

    if method == "FCFS":
        for p in sorted(processes, key=lambda x: x[1]):
            time = max(time, p[1])
            time += p[2]
            completion[p[0]] = time
            order.append(p[0])

    elif method in ["SJF", "Priority"]:
        done = set()

        while len(done) < n:
            ready = [p for p in processes
                     if p[0] not in done and p[1] <= time]

            if not ready:
                time = min(p[1] for p in processes
                           if p[0] not in done)
                continue

            if method == "SJF":
                p = min(ready, key=lambda x: x[2])
            else:
                p = min(ready, key=lambda x: x[3])

            time += p[2]
            completion[p[0]] = time
            done.add(p[0])
            order.append(p[0])

    else:
        from collections import deque

        pending = sorted(processes, key=lambda x: x[1])
        q = deque()
        i = 0

        while len(completion) < n:
            while i < n and pending[i][1] <= time:
                q.append(pending[i])
                i += 1

            if not q:
                time = pending[i][1]
                continue

            p = q.popleft()
            run = min(quantum, remaining[p[0]])

            time += run
            remaining[p[0]] -= run
            order.append(p[0])

            while i < n and pending[i][1] <= time:
                q.append(pending[i])
                i += 1

            if remaining[p[0]] > 0:
                q.append(p)
            else:
                completion[p[0]] = time

    print("\n", method)
    print("Process\tWaiting Time\tTurnaround Time")

    for p in processes:
        pid, at, bt, priority = p
        tat = completion[pid] - at
        wt = tat - bt
        print(pid, "\t", wt, "\t\t", tat)

    print("Execution Order:", order)


processes = [
    ["P1", 0, 7, 2],
    ["P2", 2, 4, 1],
    ["P3", 4, 1, 3],
    ["P4", 5, 4, 2]
]

calculate(processes, "FCFS")
calculate(processes, "SJF")
calculate(processes, "Priority")
calculate(processes, "Round Robin", 2)


 FCFS
Process	Waiting Time	Turnaround Time
P1 	 0 		 7
P2 	 5 		 9
P3 	 7 		 8
P4 	 7 		 11
Execution Order: ['P1', 'P2', 'P3', 'P4']

 SJF
Process	Waiting Time	Turnaround Time
P1 	 0 		 7
P2 	 6 		 10
P3 	 3 		 4
P4 	 7 		 11
Execution Order: ['P1', 'P3', 'P2', 'P4']

 Priority
Process	Waiting Time	Turnaround Time
P1 	 0 		 7
P2 	 5 		 9
P3 	 11 		 12
P4 	 6 		 10
Execution Order: ['P1', 'P2', 'P4', 'P3']

 Round Robin
Process	Waiting Time	Turnaround Time
P1 	 9 		 16
P2 	 3 		 7
P3 	 2 		 3
P4 	 6 		 10
Execution Order: ['P1', 'P2', 'P1', 'P3', 'P2', 'P4', 'P1', 'P4', 'P1']


In [5]:
import threading
import time

buffer = []
SIZE = 5

empty = threading.Semaphore(SIZE)
full = threading.Semaphore(0)
mutex = threading.Semaphore(1)

def producer():
    for i in range(1, 6):
        empty.acquire()
        mutex.acquire()

        buffer.append(i)
        print("Produced:", i)

        mutex.release()
        full.release()
        time.sleep(1)

def consumer():
    for i in range(1, 6):
        full.acquire()
        mutex.acquire()

        item = buffer.pop(0)
        print("Consumed:", item)

        mutex.release()
        empty.release()
        time.sleep(1)

p = threading.Thread(target=producer)
c = threading.Thread(target=consumer)

p.start()
c.start()

p.join()
c.join()

print("Producer Consumer completed.")

Produced: 1
Consumed: 1
Produced: 2
Consumed: 2
Produced: 3
Consumed: 3
Produced: 4
Consumed: 4
Produced: 5
Consumed: 5
Producer Consumer completed.


In [6]:
import threading
import time

read_count = 0

mutex = threading.Semaphore(1)
wrt = threading.Semaphore(1)

def reader(num):
    global read_count

    mutex.acquire()
    read_count += 1

    if read_count == 1:
        wrt.acquire()

    mutex.release()

    print("Reader", num, "is reading")
    time.sleep(1)

    mutex.acquire()
    read_count -= 1

    if read_count == 0:
        wrt.release()

    mutex.release()

    print("Reader", num, "finished reading")

def writer(num):
    wrt.acquire()

    print("Writer", num, "is writing")
    time.sleep(1)

    print("Writer", num, "finished writing")

    wrt.release()

r1 = threading.Thread(target=reader, args=(1,))
r2 = threading.Thread(target=reader, args=(2,))
w1 = threading.Thread(target=writer, args=(1,))

r1.start()
r2.start()
w1.start()

r1.join()
r2.join()
w1.join()

print("Readers-Writers completed.")

Reader 1 is reading
Reader 2 is reading
Reader 1 finished reading
Reader 2 finished reading
Writer 1 is writing
Writer 1 finished writing
Readers-Writers completed.


In [7]:
import threading
import time

N = 5

forks = [threading.Semaphore(1) for i in range(N)]

room = threading.Semaphore(4)

def philosopher(i):

    for j in range(2):

        print("Philosopher", i, "is thinking")
        time.sleep(1)

        room.acquire()

        forks[i].acquire()
        forks[(i + 1) % N].acquire()

        print("Philosopher", i, "is eating")
        time.sleep(1)

        forks[i].release()
        forks[(i + 1) % N].release()

        room.release()

        print("Philosopher", i, "finished eating")

threads = []

for i in range(N):
    t = threading.Thread(target=philosopher, args=(i,))
    threads.append(t)
    t.start()

for t in threads:
    t.join()

print("Dining Philosophers completed.")

Philosopher 0 is thinking
Philosopher 1 is thinking
Philosopher 2 is thinking
Philosopher 3 is thinking
Philosopher 4 is thinking
Philosopher 0 is eating
Philosopher 2 is eating
PhilosopherPhilosopher 4 is eating
 0 finished eating
Philosopher 0 is thinking
Philosopher 2 finished eating
Philosopher 2 is thinking
Philosopher 1 is eating
Philosopher 4 finished eating
Philosopher 4 is thinking
Philosopher 3 is eating
Philosopher 1 finished eating
Philosopher 1 is thinking
Philosopher 0 is eating
PhilosopherPhilosopher 2 is eating
 3 finished eating
Philosopher 3 is thinking
Philosopher 0 finished eating
Philosopher 4 is eating
Philosopher 2 finished eating
Philosopher 1 is eating
Philosopher 4 finished eating
Philosopher 3 is eating
Philosopher 3 finished eating
Philosopher 1 finished eating
Dining Philosophers completed.


In [8]:
def bankers_algorithm():

    allocation = [
        [0, 1, 0],
        [2, 0, 0],
        [3, 0, 2],
        [2, 1, 1],
        [0, 0, 2]
    ]

    maximum = [
        [7, 5, 3],
        [3, 2, 2],
        [9, 0, 2],
        [2, 2, 2],
        [4, 3, 3]
    ]

    available = [3, 3, 2]

    n = len(allocation)
    m = len(available)

    need = []

    for i in range(n):
        row = []
        for j in range(m):
            row.append(maximum[i][j] - allocation[i][j])
        need.append(row)

    print("Need Matrix:")

    for row in need:
        print(row)

    finish = [False] * n
    work = available[:]
    safe_sequence = []

    while len(safe_sequence) < n:

        found = False

        for i in range(n):

            if not finish[i]:

                if all(need[i][j] <= work[j] for j in range(m)):

                    for j in range(m):
                        work[j] += allocation[i][j]

                    safe_sequence.append("P" + str(i))
                    finish[i] = True
                    found = True

        if not found:
            break

    if all(finish):
        print("\nSystem is in SAFE state")
        print("Safe Sequence:", safe_sequence)

    else:
        print("\nSystem is in UNSAFE state")


bankers_algorithm()

Need Matrix:
[7, 4, 3]
[1, 2, 2]
[6, 0, 0]
[0, 1, 1]
[4, 3, 1]

System is in SAFE state
Safe Sequence: ['P1', 'P3', 'P4', 'P0', 'P2']


In [9]:
def memory_allocation(blocks, processes, method):

    free = blocks.copy()
    allocation = [-1] * len(processes)

    for i in range(len(processes)):

        candidates = [
            j for j in range(len(free))
            if free[j] >= processes[i]
        ]

        if not candidates:
            continue

        if method == "First Fit":
            index = candidates[0]

        elif method == "Best Fit":
            index = min(candidates, key=lambda j: free[j])

        elif method == "Worst Fit":
            index = max(candidates, key=lambda j: free[j])

        allocation[i] = index
        free[index] -= processes[i]

    print("\n", method)
    print("Process Size\tBlock Number")

    for i in range(len(processes)):

        if allocation[i] == -1:
            print(processes[i], "\t\tNot Allocated")

        else:
            print(processes[i], "\t\t", allocation[i] + 1)

    print("Remaining Memory:", free)


blocks = [100, 500, 200, 300, 600]

processes = [212, 417, 112, 426]

memory_allocation(blocks, processes, "First Fit")
memory_allocation(blocks, processes, "Best Fit")
memory_allocation(blocks, processes, "Worst Fit")


 First Fit
Process Size	Block Number
212 		 2
417 		 5
112 		 2
426 		Not Allocated
Remaining Memory: [100, 176, 200, 300, 183]

 Best Fit
Process Size	Block Number
212 		 4
417 		 2
112 		 3
426 		 5
Remaining Memory: [100, 83, 88, 88, 174]

 Worst Fit
Process Size	Block Number
212 		 5
417 		 2
112 		 5
426 		Not Allocated
Remaining Memory: [100, 83, 200, 300, 276]


In [10]:
def fifo(pages, capacity):

    frames = []
    faults = 0

    for page in pages:

        if page not in frames:
            faults += 1

            if len(frames) < capacity:
                frames.append(page)

            else:
                frames.pop(0)
                frames.append(page)

        print("Page:", page, "Frames:", frames)

    print("Total Page Faults:", faults)


def lru(pages, capacity):

    frames = []
    recent = []
    faults = 0

    for page in pages:

        if page not in frames:
            faults += 1

            if len(frames) < capacity:
                frames.append(page)

            else:
                old = recent.pop(0)
                frames.remove(old)
                frames.append(page)

        else:
            recent.remove(page)

        recent.append(page)

        print("Page:", page, "Frames:", frames)

    print("Total Page Faults:", faults)


def optimal(pages, capacity):

    frames = []
    faults = 0

    for i in range(len(pages)):

        page = pages[i]

        if page not in frames:

            faults += 1

            if len(frames) < capacity:
                frames.append(page)

            else:
                future = pages[i + 1:]

                positions = []

                for f in frames:
                    if f in future:
                        positions.append(future.index(f))
                    else:
                        positions.append(float('inf'))

                index = positions.index(max(positions))
                frames[index] = page

        print("Page:", page, "Frames:", frames)

    print("Total Page Faults:", faults)


pages = [7, 0, 1, 2, 0, 3, 0, 4, 2, 3, 0, 3, 2]

capacity = 3

print("\nFIFO Algorithm")
fifo(pages, capacity)

print("\nLRU Algorithm")
lru(pages, capacity)

print("\nOptimal Algorithm")
optimal(pages, capacity)


FIFO Algorithm
Page: 7 Frames: [7]
Page: 0 Frames: [7, 0]
Page: 1 Frames: [7, 0, 1]
Page: 2 Frames: [0, 1, 2]
Page: 0 Frames: [0, 1, 2]
Page: 3 Frames: [1, 2, 3]
Page: 0 Frames: [2, 3, 0]
Page: 4 Frames: [3, 0, 4]
Page: 2 Frames: [0, 4, 2]
Page: 3 Frames: [4, 2, 3]
Page: 0 Frames: [2, 3, 0]
Page: 3 Frames: [2, 3, 0]
Page: 2 Frames: [2, 3, 0]
Total Page Faults: 10

LRU Algorithm
Page: 7 Frames: [7]
Page: 0 Frames: [7, 0]
Page: 1 Frames: [7, 0, 1]
Page: 2 Frames: [0, 1, 2]
Page: 0 Frames: [0, 1, 2]
Page: 3 Frames: [0, 2, 3]
Page: 0 Frames: [0, 2, 3]
Page: 4 Frames: [0, 3, 4]
Page: 2 Frames: [0, 4, 2]
Page: 3 Frames: [4, 2, 3]
Page: 0 Frames: [2, 3, 0]
Page: 3 Frames: [2, 3, 0]
Page: 2 Frames: [2, 3, 0]
Total Page Faults: 9

Optimal Algorithm
Page: 7 Frames: [7]
Page: 0 Frames: [7, 0]
Page: 1 Frames: [7, 0, 1]
Page: 2 Frames: [2, 0, 1]
Page: 0 Frames: [2, 0, 1]
Page: 3 Frames: [2, 0, 3]
Page: 0 Frames: [2, 0, 3]
Page: 4 Frames: [2, 4, 3]
Page: 2 Frames: [2, 4, 3]
Page: 3 Frames: [2, 4, 3

In [11]:
def fcfs(requests, head):

    movement = 0
    current = head

    for track in requests:
        movement += abs(current - track)
        current = track

    print("FCFS Order:", requests)
    print("Total Head Movement:", movement)


def sstf(requests, head):

    pending = requests.copy()
    current = head
    movement = 0
    order = []

    while pending:

        nearest = min(pending, key=lambda x: abs(x - current))

        movement += abs(current - nearest)
        current = nearest

        order.append(nearest)
        pending.remove(nearest)

    print("SSTF Order:", order)
    print("Total Head Movement:", movement)


def scan(requests, head, size):

    left = sorted([x for x in requests if x < head])
    right = sorted([x for x in requests if x >= head])

    order = right + [size - 1] + left[::-1]

    movement = 0
    current = head

    for track in order:
        movement += abs(current - track)
        current = track

    print("SCAN Order:", order)
    print("Total Head Movement:", movement)


def cscan(requests, head, size):

    left = sorted([x for x in requests if x < head])
    right = sorted([x for x in requests if x >= head])

    order = right + [size - 1, 0] + left

    movement = 0
    current = head

    for track in order:
        movement += abs(current - track)
        current = track

    print("C-SCAN Order:", order)
    print("Total Head Movement:", movement)


requests = [82, 170, 43, 140, 24, 16, 190]

head = 50
disk_size = 200

print("\nFCFS Algorithm")
fcfs(requests, head)

print("\nSSTF Algorithm")
sstf(requests, head)

print("\nSCAN Algorithm")
scan(requests, head, disk_size)

print("\nC-SCAN Algorithm")
cscan(requests, head, disk_size)


FCFS Algorithm
FCFS Order: [82, 170, 43, 140, 24, 16, 190]
Total Head Movement: 642

SSTF Algorithm
SSTF Order: [43, 24, 16, 82, 140, 170, 190]
Total Head Movement: 208

SCAN Algorithm
SCAN Order: [82, 140, 170, 190, 199, 43, 24, 16]
Total Head Movement: 332

C-SCAN Algorithm
C-SCAN Order: [82, 140, 170, 190, 199, 0, 16, 24, 43]
Total Head Movement: 391


In [12]:
def contiguous_allocation(start, length):

    print("\nContiguous Allocation")

    blocks = list(range(start, start + length))

    print("File Blocks:", blocks)
    print("Starting Block:", start)
    print("File Length:", length)


def linked_allocation(blocks):

    print("\nLinked Allocation")

    for i in range(len(blocks)):

        if i == len(blocks) - 1:
            next_block = "NULL"
        else:
            next_block = blocks[i + 1]

        print("Block", blocks[i], "->", next_block)


def indexed_allocation(index_block, blocks):

    print("\nIndexed Allocation")

    print("Index Block:", index_block)

    for i in range(len(blocks)):
        print("Index", i, "-> Data Block", blocks[i])


contiguous_allocation(10, 5)

linked_allocation([7, 19, 4, 25, 11])

indexed_allocation(30, [7, 19, 4, 25, 11])


Contiguous Allocation
File Blocks: [10, 11, 12, 13, 14]
Starting Block: 10
File Length: 5

Linked Allocation
Block 7 -> 19
Block 19 -> 4
Block 4 -> 25
Block 25 -> 11
Block 11 -> NULL

Indexed Allocation
Index Block: 30
Index 0 -> Data Block 7
Index 1 -> Data Block 19
Index 2 -> Data Block 4
Index 3 -> Data Block 25
Index 4 -> Data Block 11
